In [208]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LeakyReLU
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from joblib import dump, load

import torch
import torch.nn as nn

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

./Project_3.csv
./Project_3.ipynb
./.ipynb_checkpoints/Project_3-checkpoint.ipynb
./.ipynb_checkpoints/Project_3-checkpoint.csv
./data/hello
./data/.ipynb_checkpoints/hello-checkpoint


# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

We wish to predict whether a given person at our customer's shop will buy a widget based on their account details. The attributes tied to each person's account are their User ID, name, gender, age, estimated salary, and whether they bought or returned a widget or did not do either. Our goal is to train a model based on this data that can predict whether a widget was bought, still containing a return as a sale, or was not bought based on a person's attributes.

An important aspect of this project is that we know beforehand that the data behaves highly unusually, making it difficult to pick up on patterns to make predictions.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

We were provided the data directly by the customer. In this instance, we were given data that displayed similar patterns to a real problem encountered in the past.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [44]:
df_explore = pd.read_csv('Project_3.csv')
df_explore['Purchased'] = df['Purchased'] ** 2
    # We make returns count the same as purchases, since for our purposes we will not distinguish between these

X_gender = df_explore[['Gender']] == 'Male'
X_age = df_explore[['Age']]
X_salary = df_explore[['EstimatedSalary']]
X_id = df_explore[['User ID']]
Y = df_explore['Purchased']
    # Set up X and Y sets to run linear regressions on

model_gender = LinearRegression()
model_gender.fit(X_gender, Y)
model_age = LinearRegression()
model_age.fit(X_age, Y)
model_salary = LinearRegression()
model_salary.fit(X_salary, Y)
model_id = LinearRegression()
model_id.fit(X_id, Y)
    # Run linear regression
print('Gender', model_gender.coef_[0], model_gender.intercept_, model_gender.score(X_gender, Y))
print('Age', model_age.coef_[0], model_age.intercept_, model_age.score(X_age, Y))
print('Salary', model_salary.coef_[0], model_salary.intercept_, model_salary.score(X_salary, Y))
print('ID', model_id.coef_[0], model_id.intercept_, model_id.score(X_id, Y))
    # Print results of regressions

Gender -0.029549902152641833 0.5009784735812133 0.0008734478784081512
Age -0.0016944508079668004 0.5528679093121784 0.0014896421100468737
Salary 8.056516325864068e-07 0.42329146522251554 0.0045960539976900305
ID -1.0899280360358191e-05 0.6009559302972476 3.970618833892825e-05


Given a general lack of features, we found the correlations between every attribute besides name with purchase status. We first remark that we were biased towards estimated salary being the strongest predictor of purchase status. However, every r^2 value was less than 0.002, which implies an almost negligible positive or negative correlation for each of the features. This means that if there are patterns in the data, they are most likely nonlinear and hard to find. When choosing a model, we should keep in mind that in needs to be able to pick up on such hard to find patterns.

In [188]:
p = (df_explore['Purchased'] == 1).sum() / len(df_explore)
q = (df_explore['Purchased'] == 0).sum() / len(df_explore)
r = (df_explore['Purchased'] == -1).sum() / len(df_explore)
    # Proportions of purchases
print(p, q, r)
print(f"Accuracy by predicting the same value for each row: {max(p, q, r)}")
print(f"Accuracy by randomly predicting purchase with probabilities: {p ** 2 + q ** 2 + r ** 2}")

0.41858141858141856 0.5134865134865135 0.06793206793206794
Accuracy by predicting the same value for each row: 0.5134865134865135
Accuracy by randomly predicting purchase with probabilities: 0.4434935693676953


Before moving on to finding a model, given that we have picked up on no patterns, we decided to check the average behavior of the 'Purchased' column. This was to see if the proportion of purchases differed significantly from 0.5, but also to calculate what baseline accuracy we should look for to see if our model is actually performing better than chance.

We did find that the proportion of purchases was significantly lower than 0.5. We also found that we should attempt to find a model that is correct at least around 51% of the time to beat just predicting the mode.

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [60]:
df = pd.read_csv('Project_3.csv')

df['Skipped'] = df['Purchased'] == 0
df['Returned'] = df['Purchased'] == -1
df['Purchased'] = df['Purchased'] == 1
    # We received a hint to separate the last column into 3 different columns for the 3 categories it represents

df.drop(['User ID', 'name'], axis=1, inplace=True)
    # 'name' is dropped because we believe it will not be helpful and it is hard to work with

df['Gender'] = df['Gender'] == 'Male'
    # 'Gender' is converted into a binary value because all rows are either Male or Female

print(df.head(20))

    Gender  Age  EstimatedSalary  Purchased  Skipped  Returned
0    False   42            74248       True    False     False
1     True   49           122650       True    False     False
2     True   25            18499      False     True     False
3    False   36           112622      False    False      True
4     True   42           148223      False     True     False
5    False   26            69489       True    False     False
6    False   38            39268       True    False     False
7     True   49            36599      False     True     False
8     True   44            74145      False     True     False
9    False   30            41319       True    False     False
10   False   30           118234      False     True     False
11    True   39           110305      False     True     False
12    True   50            68251      False     True     False
13   False   36            42239      False     True     False
14   False   32           125017      False     True   

In [131]:
X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [201]:
    # Model using KNN

X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
y = 1 * y[y.columns[1]] + 2 * y[y.columns[2]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)
model_knn = KNeighborsClassifier(n_neighbors=20)
model_knn.fit(X_train_normalized, y_train)
y_pred_knn = model_knn.predict(X_test_normalized)
accuracy = accuracy_score(y_test, y_pred_knn)
print(accuracy)

0.49800796812749004


In [190]:
    # Model using neural network

X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

model_neural = Sequential([
    Input(shape=(X_train_normalized.shape[1],)),
    Dense(32),
    LeakyReLU(negative_slope=0.01),
    Dense(16),
    LeakyReLU(negative_slope=0.01),
    Dense(3, activation='softmax')
])
model_neural.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history = model_neural.fit(
    X_train_normalized,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0,
    verbose=0
)
y_pred = model_neural.predict(X_test_normalized)
y_pred_neural = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)
accuracy = accuracy_score(y_test_classes, y_pred_neural)
print(accuracy)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
0.4302788844621514


In [191]:
    # Model using random forests

X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
y = y[y.columns[0]] + 2 * y[y.columns[1]] + 3 * y[y.columns[2]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_rf)
print(accuracy)

0.3904382470119522


In [192]:
    # Model using Gaussian Naive Bayes

X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
y = y[y.columns[0]] + 2 * y[y.columns[1]] + 3 * y[y.columns[2]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

model_gnb = GaussianNB()
model_gnb.fit(X_train, y_train)
y_pred_gnb = model_gnb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_gnb)
print(accuracy)

0.4820717131474104


We start with no discernible patterns in the data. We first tried a KNN model to see if there was any easy-to-pick-up-on clustering. However, its accuracy of 49.8% was worse than just predicting the mode (and in fact less than 50%). We then received a hint to use a neural network, but that yielded a lower accuracy of 43.0%. We then looked at random forests, since for previous projects this model has been successful, but it yielded the lowest accuracy of 39.0%. Finally, we looked at a Gaussian Naive Bayes model, since a classmate (Kyra) told me she was successful with Bernoulli Naive Bayes, although I used Gaussians to avoid having to binarize continuous data. It yielded an accuracy of 48.2%, similar to KNN. Since the highest accuracy was given by KNN, we will move forward with KNN.

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


In [220]:
    # Parameter tuning with KNN

X = df[df.columns[:-3]]
y = df[df.columns[-3:]]
y = 1 * y[y.columns[1]] + 2 * y[y.columns[2]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

param_grid = {
    'n_neighbors': list(range(1, 51)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1, 2]
}
model_knn = KNeighborsClassifier()
grid_search = GridSearchCV(model_knn, param_grid, scoring='accuracy')
grid_search.fit(X_train, y_train)
final_model = grid_search.best_estimator_
dump(final_model, 'final_model.joblib')
y_pred = final_model.predict(X_test)
y_pred[y_pred == 2] = 0
y_test.replace(2, 0)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.5059760956175299


We defined a grid of parameters to search through to fine tune the parameters for our KNN model. Finally, at this point we recall that we made the distinction between Purchases and Returns because we believed it was useful information. Since we originally stated that they were to be treated the same, we now convert all Returns to Purchases before computing the final accuracy, which comes out to 50.6%.

Small note: I originally forgot to include feature scaling. However, adding it back lowered the accuracy, so it was omitted for the final model.

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


The given problem was to create a model that could help our customer in predicting whether a given person would buy a widget at their shop. We were told (given the context of the problem) that the data behaved unusually, and we also observed this by looking at the data. When exploring the data, we were not able to find any strong patterns either by direct inspection or by running statistical procedures. When preparing our data, we dropped Names because they would be hard to work with, as well as User ID as we believed that ID would not be helpful, leaving us with Gender, Age, and Salary to make predictions. We also separated the Purchase column into 3 separate columns for Purchase, Not Purchase, and Return, as we believed this separating into the 3 distinct variables that this column actually represents would make it easier to train a model. Since we started with no patterns, we tested 4 types of models to have the best chance of one of them picking up on some pattern, namely KNN, neural networks, random forests, and Gaussian naive Bayes. Each of these models yielded relatively low accuracies, but among them KNN yielded the highest, and so we moved forward with KNN for parameter tuning. We did not get a significant increase from parameter tuning, and ended up with 50.6% accuracy. We note that in the final accuracy calculation, we treated Returns and Purchases to be the same, since the distinction was only made to make it easier to pick up on patterns.

The final accuracy of 50.6% is worse than just predicting the mode value. So, we were not successful in creating a strong model to solve the customer's problem. We were told that the data exhibited strange patterns that may be hard to find, but we were not able to create a model that successfully picked up on any such patterns.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [225]:
def infrence(gender: str, Age: float, EstimatedSalary: float) -> str:
        # Gender must be either 'Male' or 'Female'
    if gender not in ['Male', 'Female']:
        raise Exception("The parameter gender must be either 'Male' or 'Female'")
    import os
    import pandas as pd
    from sklearn.neighbors import KNeighborsClassifier
    from joblib import load
    import numpy as np
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)
    model = load('final_model.joblib')
    input_attributes = [[gender == 'Male', Age, EstimatedSalary]]
    prediction = model.predict(input_attributes)[0]
    if prediction in [0, 2]:
        return 'Purchase'
    else:
        return 'Not Purchase'

In [226]:
print(infrence('Male', 20, 100000))

Not Purchase
